## III. `calc_total_score()` → score, p ⬜ 🧪
Combines all scores of the remaining recipes (after the hard filter) and draws the proposals.

> Uses: date_score ([date_score.ipynb](date_score.ipynb)) | reward_score ([reward_score.ipynb](reward_score.ipynb)) | damp_score_mult / damp_score_offset ([damping.ipynb](damping.ipynb)) | wishlist

`wishlist_score` = 10 if on the wishlist, else 0.

$$
\text{score} = (\text{date\_score} + \text{reward\_score} + \text{wishlist\_score}) \cdot \text{damp\_score\_mult} + \text{damp\_score\_offset} + \frac{\text{wishlist\_score}}{5}
$$

The wishlist 10 is damped, the extra `wishlist_score / 5` (+2) is added undamped after damping.

### Two pools
After the hard filter the recipes are split:

| Pool | Recipes | Score |
|---|---|---|
| **main** | cooked at least once, or on the wishlist | formula above |
| **new** | never cooked (and not on the wishlist) | formula above with `date_score = 5` and the reward from the recipe's times + midpoint ratings (taste 5, effort 5); no times → **median** of all computed scores |

- Proposals are drawn from the **main** pool. Fewer than 5 left → filled up with draws from the **new** pool.
- Swapping (20 % of all lists) draws the replacement from the **new** pool first, then from the rest.
- Website button **"Neue Rezepte (nie gekocht)"** draws only from the **new** pool.
- Nothing left at all → message with "browse recipes" and "reroll with more randomness" (`T + 1`).
- The wishlist overrides `remove` from date_score (but not the blacklist).

The demo values below are hand-picked inputs, not computed by the other notebooks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "id":                [3, 5, 6, 7, 8, 9],
    "name":              ["Linsensuppe", "Apfelstrudel", "Lachs-Pasta", "Burrito", "Curry", "Risotto"],
    "date_score":        [ 6.0,  0.0, 8.0,  4.0, 9.0, 1.0],      # date_score ist auf [0, 10] geclippt
    "reward_score":      [ 5.5,  3.0, 6.2,  7.0, 4.8, 6.5],
    "damp_score_mult":   [ 1.0,  0.2, 1.0, 0.72, 1.0, 0.9],      # Burrito wie in damping.ipynb
    "damp_score_offset": [ 0.0,  2.0, -3.0, -3.0, 3.0, 0.0],
})
wishlist = {6, 9}          # IDs auf der Wunschliste

In [2]:
def wishlist_score(ids, wishlist):
    return np.where(np.isin(ids, list(wishlist)), 10, 0)

def total_score(date_score, reward_score, wish, mult, offset):
    return (date_score + reward_score + wish) * mult + offset + wish / 5

df["wishlist_score"] = wishlist_score(df["id"], wishlist)
df["score"] = total_score(df["date_score"], df["reward_score"], df["wishlist_score"],
                          df["damp_score_mult"], df["damp_score_offset"])
df

,id,name,date_score,reward_score,damp_score_mult,damp_score_offset,wishlist_score,score
0,3,Linsensuppe,6.0,5.5,1.00,0.0,0,11.50
1,5,Apfelstrudel,-2.0,3.0,0.20,2.0,0,2.20
2,6,Lachs-Pasta,8.0,6.2,1.00,-3.0,10,23.20
3,7,Burrito,4.0,7.0,0.63,-5.0,0,1.93
4,8,Curry,9.0,4.8,1.00,3.0,0,16.80
5,9,Risotto,1.0,6.5,0.90,0.0,10,17.75


### Softmax
Turns the scores into draw probabilities:

$$
p_i = \frac{e^{score_i / T}}{\sum_j e^{score_j / T}}
$$

Small `T` → the best recipes almost always win, large `T` → more random. Negative scores need no clamp: `exp` is always > 0.
`T` = 4 by default, the user can set it from 1 to 10 (filter "Zufall (T)" on the website, reroll = `T + 1`).

**Drawing** (here only single draws): repeat 5 times – softmax over the recipes still in the container, draw 1 without putting back, remove it **and its variants**.

**Swapping**: each proposal is swapped with probability 1 − p, p = 0.8^(1/5) ≈ 0.956 → in 20 % of all lists at least one recipe gets swapped. Wishlist recipes are never swapped.

In [ ]:
def softmax(score, T=1.0):
    z = np.asarray(score, dtype=float) / T
    z -= z.max()                     # verhindert Überlauf bei großen Scores, Ergebnis bleibt gleich
    e = np.exp(z)
    return e / e.sum()

T = 4.0                              # Standard (ProposeLogic.SOFTMAX_T), einstellbar 1-10
df["p"] = softmax(df["score"], T)
df.sort_values("p", ascending=False)

In [4]:
rng = np.random.default_rng()
choice = rng.choice(df["id"], p=df["p"])
df[df["id"] == choice]

,id,name,date_score,reward_score,damp_score_mult,damp_score_offset,wishlist_score,score,p
2,6,Lachs-Pasta,8.0,6.2,1.0,-3.0,10,23.2,0.767462


In [ ]:
Ts = [1, 4, 10]
order = df.sort_values("score", ascending=False)

x = np.arange(len(order))
w = 0.8 / len(Ts)

plt.figure(figsize=(10, 5))
for i, T in enumerate(Ts):
    plt.bar(x + i * w, softmax(order["score"], T), width=w, label=f"T={T}")
plt.xticks(x + w * (len(Ts) - 1) / 2, [f"{n}\n({s:.1f})" for n, s in zip(order["name"], order["score"])])
plt.ylabel("Wahrscheinlichkeit p")
plt.title("Softmax-Verteilung für verschiedene T")
plt.legend()
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
draws = rng.choice(df["id"], size=10_000, p=softmax(df["score"], T=4))
pd.Series(draws).value_counts(normalize=True).rename("gezogen").to_frame() \
  .join(df.set_index("id")[["name", "p"]])

## Sanity Checks

Expected behavior. 🧪 = checked in [tests/test_propose_plan.py](../../tests/test_propose_plan.py) (backend, 5 draws without putting back).

| Situation | Expected | Status |
|---|---|---|
| negative score | still drawable, but rarely | 🧪 score −4: 15.5 % at T = 4 |
| higher score ⇒ higher probability | monotonically increasing | 🧪 |
| recipe and its variant in the same list | never | 🧪 0 of 2 000 |
| at least one swap in a list | 20 % | 🧪 20.0 % |
| fewer than 5 cooked recipes | filled with new recipes | 🧪 every allowed recipe gets proposed |
| button "Neue Rezepte" | only never cooked recipes | 🧪 |
| new recipe without times | median score | 🧪 |
| wishlist recipe cooked 5 days ago | still proposed (wishlist overrides `remove`) | 🧪 |

## Edge Cases – decided

| Case | Handling |
|---|---|
| temperature `T` | default 4, adjustable from 1 to 10 |
| all scores very negative / very large | works: `z -= z.max()` keeps `exp` in range |
| equal scores | equal probability |
| fewer than 5 recipes left after the hard filter | propose all that are left + fill with new recipes |
| no recipe left | message with options to browse recipes or reroll with more randomness (`T + 1`) |
| wishlist recipe removed by the hard filter (cooked ≤ 5 days ago) | wishlist overrides `remove` |
| recipe on wishlist **and** blacklist | blacklist wins (hard filter) |
| swap candidate pool empty | no swap |